# 05 Precompute Text Embeddings

Colab-ready training notebook. Run cells from top to bottom.


## Colab Data Checklist

Before training, make sure these paths exist in the Colab runtime, usually by mounting Google Drive or uploading the generated folder:

- `atlas_free_multipositive/cache/unified_jsonl/splits/train.jsonl`
- `atlas_free_multipositive/cache/unified_jsonl/splits/val.jsonl` when validation is used
- `atlas_free_multipositive/cache/unified_jsonl/text_registry.jsonl` for text embedding precompute
- `atlas_free_multipositive/cache/text_embeddings/specter_text_cache.pt` for training notebooks after notebook 05
- `atlas_free_multipositive/cache/maps/` if JSONL rows reference NIfTI atlas maps
- `atlas_free_multipositive/data/ale_caches/atlas_free_4mm_fwhm9_crop_float16.pt` if JSONL rows reference the packed PubMed ALE tensor cache

The easiest route is to put the whole repo plus generated `atlas_free_multipositive/cache/` and `atlas_free_multipositive/data/ale_caches/` on Drive, then set `PROJECT_ROOT` below.


In [ ]:
# Optional Drive mount for Colab. Uncomment if your repo/data are on Google Drive.
# from google.colab import drive
# drive.mount('/content/drive')


In [ ]:
# Install runtime dependencies in Colab. Skip if your Colab image already has them.
# `%pip install -e .` works when running from the repo root with pyproject.toml present.
%pip install -q torch pandas numpy nibabel nilearn pyyaml tqdm safetensors transformers adapters


In [ ]:
from pathlib import Path
import os, sys

# Option A: if you cloned/uploaded the repo as the current working tree, leave this alone.
# Option B: set this to your Drive repo path, e.g. Path('/content/drive/MyDrive/neurovlm').
PROJECT_ROOT = Path.cwd()

# Walk upward if this notebook starts inside atlas_free_multipositive/notebooks.
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'atlas_free_multipositive').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / 'atlas_free_multipositive').exists():
    raise FileNotFoundError('Set PROJECT_ROOT to the repo directory containing atlas_free_multipositive/.')

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('PROJECT_ROOT =', PROJECT_ROOT)
print('cwd =', Path.cwd())


In [ ]:
from pathlib import Path

paths_to_check = [
    'atlas_free_multipositive/cache/unified_jsonl/splits/train.jsonl',
    'atlas_free_multipositive/cache/unified_jsonl/splits/val.jsonl',
    'atlas_free_multipositive/cache/unified_jsonl/text_registry.jsonl',
    'atlas_free_multipositive/cache/text_embeddings/specter_text_cache.pt',
    'atlas_free_multipositive/data/ale_caches/atlas_free_4mm_fwhm9_crop_float16.pt',
]
for p in paths_to_check:
    print(f'{Path(p).exists():5}  {p}')


## Precompute SPECTER/SPECTER2 Text Embeddings
This notebook builds `specter_text_cache.pt` for later Colab training notebooks. It freezes SPECTER/SPECTER2 and saves one embedding per unique positive text.

In [ ]:
import json, torch
from pathlib import Path
from atlas_free_multipositive.training.model_wrappers import encode_texts_specter
registry = Path('atlas_free_multipositive/cache/unified_jsonl/text_registry.jsonl')
if not registry.exists():
    raise FileNotFoundError(f'Missing {registry}. Run notebook 03 first or upload unified_jsonl/text_registry.jsonl.')
rows = [json.loads(l) for l in registry.open()]
texts = [r['text'] for r in rows]
print('unique texts:', len(texts))
emb = encode_texts_specter(texts, device='cuda' if torch.cuda.is_available() else 'cpu', batch_size=16)
cache = {t: e.cpu() for t, e in zip(texts, emb)}
out = Path('atlas_free_multipositive/cache/text_embeddings/specter_text_cache.pt')
out.parent.mkdir(parents=True, exist_ok=True)
torch.save(cache, out)
print('saved', out, emb.shape)
